In [12]:
import sys
import os
import random
import torch

repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

from vae_generator import VAEGenerator

SEQ_LENGTH = 6
INPUT_DIM = SEQ_LENGTH * 20  # 120
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

# --- Generate temporary peptide data ---
random.seed(42)
peptides = list(set(
    ''.join(random.choices(AMINO_ACIDS, k=SEQ_LENGTH))
    for _ in range(500)
))
print(f"Generated {len(peptides)} unique temporary peptides")
print(f"Example sequences: {peptides[:5]}")

# --- One-hot encode ---
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def encode(sequences, seq_length):
    tensor = torch.zeros(len(sequences), seq_length * 20)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            tensor[i, j * 20 + aa_index[aa]] = 1.0
    return tensor

one_hot = encode(peptides, SEQ_LENGTH)
print(f"Tensor shape: {one_hot.shape}")  # Expected: [n, 120]

# --- Initialize model ---
vae = VAEGenerator(input_dim=INPUT_DIM, latent_dim=32, hidden_dim=128)
print(f"Device:     {vae.device}")
print(f"Input dim:  {vae.input_dim}")
print(f"Latent dim: {vae.latent_dim}")
print(f"Hidden dim: {vae.hidden_dim}")

# --- Train ---
one_hot = one_hot.to(vae.device)
vae.train_model(one_hot, epochs=300, batch_size=64, lr=1e-3)
print("Training complete!")

# --- Generate ---
generated = vae.generate_peptides(count=10)
amino_acid_set = set(AMINO_ACIDS)
unique_generated = set()
for i, pep in enumerate(generated):
    valid = all(c in amino_acid_set for c in pep) and len(pep) == SEQ_LENGTH
    status = "✓" if valid else "✗ INVALID"
    unique_generated.add(pep)
    print(f"Peptide {i+1:02d}: {pep}  {status}")

# --- Novelty + diversity check ---
training_set = set(peptides)
novel = [p for p in generated if p not in training_set]
print(f"\nNovel peptides  (not in training data): {len(novel)}/{len(generated)}")
print(f"Unique peptides (diversity check):      {len(unique_generated)}/{len(generated)}")


Generated 500 unique temporary peptides
Example sequences: ['LVHYAW', 'LLANAM', 'YFQTQV', 'YYGQLN', 'RHPMIN']
Tensor shape: torch.Size([500, 120])
Device:     cuda
Input dim:  120
Latent dim: 32
Hidden dim: 128
Training complete!
Peptide 01: TSCYTH  ✓
Peptide 02: TAQMKH  ✓
Peptide 03: TWCSTP  ✓
Peptide 04: TLQMTM  ✓
Peptide 05: TLCMTH  ✓
Peptide 06: TSQMTY  ✓
Peptide 07: YLCMTP  ✓
Peptide 08: TLQPTP  ✓
Peptide 09: TPCMTD  ✓
Peptide 10: TLQMND  ✓

Novel peptides  (not in training data): 10/10
Unique peptides (diversity check):      10/10
